这段代码实现了一个**暴力搜索最优参数的线性回归模型**，主要包含以下几个部分：

## 1. 数据准备
定义了 6 组训练数据，`x_train` 和 `y_train` 之间存在近似 20 倍的关系（如 x=1 对应 y=20，x=9 对应 y=180）。

## 2. 两个核心函数
- **`predict_fun(w,b,x)`**：线性回归的预测函数，计算 $f_{wb}(x) = wx + b$
- **`cost_fun(x,y,w,b)`**：代价函数，计算 $\frac{1}{2m}\sum(f_{wb}(x_i) - y_i)^2$，衡量预测值与真实值的差距

## 3. 暴力搜索最优参数
采用**网格搜索**策略，在 $w \in [0,25)$、$b \in [-5,10)$ 范围内以步长 0.1 和 0.5 遍历所有参数组合，记录使代价函数最小的 $w$ 和 $b$。

初始值采用第三种方法——先用 $(w=0, b=0)$ 算一个代价作为起点，后续遇到更小的值再更新。

## 4. 运行结果
```
0.003472222222222224 20.0 0.0
```
最优参数为 $w=20.0, b=0.0$，此时代价接近于 0，说明模型完美拟合了训练数据。

## 5. 模型应用
用找到的最优参数对新数据 $x=100$ 进行预测，结果为 $20 \times 100 + 0 = 2000$。

---

这段代码虽然简单，但完整展示了机器学习的三个基本步骤：**定义模型 → 训练（寻找最优参数）→ 预测**。

In [31]:
import numpy as np

x_train=np.array([1,3,6,8,9,5])
y_train=np.array([20,60,120,160,180,100])

def predict_fun(w,b,x):
    return w*x+b

def cost_fun(x,y,w,b):
    sum_cost=0
    len_x=len(x)
    for i in range(len_x):
        # x_cost=(w*x[i]+b)-y[i]
        x_cost=predict_fun(w,b,x[i])-y[i]
        predict=x_cost**2
        sum_cost+=predict
    return sum_cost/(2*len_x)

# 有三种常见的设置初始值的方法
# 1. 如果你对数据有一定了解，可以设一个“肯定比所有可能结果都大（或小）”的值。
# 2. min_cost = float('inf')   # 找最小值
# max_value = float('-inf') # 找最大值
# 3. 先拿第一组参数算个值作为起点
min_cost=cost_fun(x_train,y_train,0,0)
min_w=0
min_b=0
for i in np.arange(0,25,0.1):
    for j in np.arange(-5,10,0.5):
        predict_i=cost_fun(x_train,y_train,i,j)
        if(predict_i<min_cost):
            min_cost=predict_i
            min_w=i
            min_b=j


print(min_cost,min_w,min_b)

# 使用训练结果预测新数据
predict_new=predict_fun(min_w,min_b,100)
print(predict_new)

0.0 20.0 0.0
2000.0


## 从暴力搜索引出的常见参数优化方法

### 暴力搜索（网格搜索）
**做法**：在预设范围内按固定步长遍历所有参数组合。

**优点**：实现简单，一定能找到给定范围内的最优解。
**缺点**：参数增多时计算量指数爆炸，需要人工设定范围和步长。

---

### 随机搜索
**做法**：在参数空间内随机采样，而不是按固定网格遍历。

**优点**：相同采样次数下覆盖范围更广，高维空间中比网格搜索更高效。
**缺点**：结果不稳定，可能错过关键区域的最优解。

---

### 梯度下降
**做法**：计算代价函数对参数的梯度，沿梯度反方向迭代更新参数。

**优点**：不遍历所有组合，计算量与参数个数线性相关，适合大规模参数。
**缺点**：可能陷入局部最优，需要调学习率，要求代价函数可导。

---

### 遗传算法 / 粒子群优化
**做法**：模拟生物进化或群体智能，通过变异、交叉、选择等操作迭代寻优。

**优点**：不要求函数可导，适合离散或复杂非凸问题，不易陷入局部最优。
**缺点**：收敛速度慢，超参数多（种群大小、变异率等），调参成本高。

---

### 贝叶斯优化
**做法**：建立概率代理模型，在不确定性高的区域和已知最优区域之间权衡采样。

**优点**：采样次数少，适合代价函数计算昂贵（如深度学习调参）的场景。
**缺点**：实现复杂，在高维参数空间中效果下降明显。